# Pulse 2026: Sample Size vs IQB Score

This notebook explores whether a country's sample size (number of measurements in a given time period)
influences its resulting IQB score, and whether scores are more volatile for countries
with fewer measurements.

**Dataset:** October 2025

**Questions:**
1. Is there a correlation between sample count and IQB score?
2. Do low-sample countries show higher IQB score variance across percentile choices?
3. Is there a sample count threshold below which scores appear unreliable?

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pycountry
import pyarrow.parquet as pq

from iqb import IQBCalculator

calculator = IQBCalculator()

PERCENTILES = [25, 50, 75]
DATA_DIR = Path("./data/v1/20251001T000000Z/20251101T000000Z")

In [ ]:
dl = pq.read_table(DATA_DIR / "downloads_by_country/data.parquet").to_pandas()
ul = pq.read_table(DATA_DIR / "uploads_by_country/data.parquet").to_pandas()

# Merge on country_code; sample_count appears in both tables
merged = dl.merge(ul, on="country_code", suffixes=("_down", "_up"))

# Use the minimum of download/upload sample counts as the conservative estimate of sample size for each country
merged["sample_count"] = merged[["sample_count_down", "sample_count_up"]].min(axis=1)

# Resolve full country name from ISO-2 code; fall back to code if not found
def _country_name(cc: str) -> str:
    c = pycountry.countries.get(alpha_2=cc)
    return c.name if c else cc

merged["country_name"] = merged["country_code"].map(_country_name)

print(f"Countries: {len(merged)}")
print(f"Sample count range: {merged['sample_count'].min():,} – {merged['sample_count'].max():,}")
merged[["country_code", "country_name", "sample_count_down", "sample_count_up", "sample_count"]].head(8)

In [ ]:
def compute_iqb(row: pd.Series, p: int) -> float:
    data = {
        "m-lab": {
            "download_throughput_mbps": row[f"download_p{p}"],
            "upload_throughput_mbps": row[f"upload_p{p}"],
            "latency_ms": row[f"latency_p{p}"],
            "packet_loss": row[f"loss_p{p}"],
        }
    }
    return calculator.calculate_iqb_score(data=data)


for p in PERCENTILES:
    merged[f"iqb_p{p}"] = merged.apply(compute_iqb, p=p, axis=1)

iqb_cols = [f"iqb_p{p}" for p in PERCENTILES]
merged["iqb_range"] = merged[iqb_cols].max(axis=1) - merged[iqb_cols].min(axis=1)
merged["iqb_std"] = merged[iqb_cols].std(axis=1)

print("IQB scores computed. Sample rows:")
merged[["country_code", "sample_count"] + iqb_cols + ["iqb_range"]].head(8)


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

log_counts = np.log10(merged["sample_count"].clip(lower=1))
bins = np.linspace(log_counts.min(), log_counts.max(), 30)
ax.hist(log_counts, bins=bins, color="steelblue", edgecolor="white", linewidth=0.4)

ax.set_xlabel("Sample count (log₁₀ scale)", fontsize=11)
ax.set_ylabel("Number of countries", fontsize=11)
ax.set_title("Distribution of M-Lab sample counts per country (Oct 2025)", fontsize=12)

ax.set_xticks([1,2,3,4,5,6,7])
ax.set_xticklabels([f"10^{v}" for v in range(1, 8)])
ax.set_yticks([0, 5, 10, 15, 20])

median_log = np.median(log_counts)
ax.axvline(median_log, color="tomato", linestyle="--", linewidth=1.5,
           label=f"Median: {10**median_log:,.0f}")
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

print("\nSample size quantile breakdown:")
for q in [0.1, 0.25, 0.5, 0.75, 0.9, 0.95]:
    v = merged["sample_count"].quantile(q)
    print(f"  P{int(q*100):3d}: {v:>12,.0f} measurements")